In [12]:
"""
S3 Sales Data Download Script

- Connects to AWS S3 using credentials from a YAML file
- Downloads sales data from an S3 prefix
- Supports:
  1. Full snapshot load
  2. Full load partitioned by ingestion date
"""


import boto3
import yaml
from pathlib import Path
from datetime import date
import os


# Configuration
S3_BUCKET = 'cm-aws-s3-data-source'
S3_FOLDER_PATH = 'organization/sales'

# Load AWS Credentials
# Assumes credentials.yml is in the project root directory
parent_cwd = Path.cwd().parent
files = list(parent_cwd.glob(pattern="credentials.yml"))

# Read yaml file
with open('credentials.yml', 'r') as f:
    credentials = yaml.safe_load(f)

    aws_access_key_id = credentials["aws"]["aws_access_key_id"]
    aws_secret_access_key = credentials["aws"]["aws_secret_access_key"]

# Create S3 Client
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)

s3 = session.client(service_name='s3')

Full Load - Snapshot

In [13]:
# List all objects under the S3 sales prefix
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_FOLDER_PATH)
objects = response.get('Contents', [])

# Local destination for snapshot load
destination_folder = "destination/sales/snapshot"
os.makedirs(destination_folder, exist_ok=True)

# Download each file from S3
for obj in objects:
    s3_key = obj['Key']

    # Skip S3 folder placeholders
    if s3_key.endswith('/'):
        continue

    file_name = s3_key.split('/')[-1]

    local_file_path = os.path.join(destination_folder, file_name)

    try:
        s3.download_file(Bucket=S3_BUCKET, Key=s3_key, Filename=local_file_path)
        print(f"Downloaded: {file_name}")
    except Exception as e:
        print(f"ERROR: Failed to download {file_name}. Reason: {e}")
        continue   

Downloaded: coffee_sales_202403.csv
Downloaded: coffee_sales_202404.csv
Downloaded: coffee_sales_202405.csv
Downloaded: coffee_sales_202406.csv
Downloaded: coffee_sales_202407.csv
Downloaded: coffee_sales_202408.csv
Downloaded: coffee_sales_202409.csv
Downloaded: coffee_sales_202410.csv
Downloaded: coffee_sales_202411.csv
Downloaded: coffee_sales_202412.csv


Full Load - Partition by Ingestion Date

In [14]:
# Use today's date as folder name for partitioned load
ingestion_date = date.today().isoformat()
destination_folder = f"destination/sales/{ingestion_date}"
os.makedirs(destination_folder, exist_ok=True)

# Download each file from S3
for obj in objects:
    s3_key = obj['Key']  

    # Skip S3 folder placeholders            
    if s3_key.endswith('/'):         
        continue

    file_name = s3_key.split('/')[-1]                     
    local_file_path = os.path.join(destination_folder, file_name) 

    try:
        s3.download_file(Bucket=S3_BUCKET, Key=s3_key, Filename=local_file_path)
        print(f"Downloaded: {file_name} to {destination_folder}")
    except Exception as e:
        print(f"ERROR: Failed to download {file_name}. Reason: {e}")

Downloaded: coffee_sales_202403.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202404.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202405.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202406.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202407.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202408.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202409.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202410.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202411.csv to destination/sales/2026-01-18
Downloaded: coffee_sales_202412.csv to destination/sales/2026-01-18
